# IOE 424 – GM ECR Analytics
## Data Cleaning and Preliminary Analysis

**Objective:** Clean the raw ECR data file provided by GM and do a first round of exploratory analysis.

*This notebook was written for GM's first (unmasked, 28-column) sample extract. Put that file in `data/` to run it; the main pipeline (`run_pipeline.py`) uses the later masked extract.*

**Steps:**
1. Import libraries and set parameters
2. Load the raw data
3. Clean the data (text, data types, delimiters to lists, new columns)
4. Filter to Closed ECRs
5. Partition by system: Legacy vs. EMERGE (pre/post 2024)
6. Partition by MODEL_YEAR
7. Analyze CREATED_ON vs. MODEL_YEAR
8. Export the cleaned data
9. Summary of results

## Step 1: Import Libraries and Set Parameters

In [ ]:
import re
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 80)

The parameters below are used throughout the notebook. A few of them are assumptions we still need to confirm with GM:
- **EMERGE_WO_THRESHOLD:** WO numbers fall into two ranges (about 4.1M–4.8M and 7.0M+). We assume the 7M+ range is EMERGE. Step 5 checks this.
- **MY_PLACEHOLDERS:** MODEL_YEAR 2035 has no records near it (nothing in 2032–2034), so we treat it as a placeholder.
- **MY_START_MONTH / OFFSET:** We assume model year N starts production around July of year N-1.

In [ ]:
# File paths
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = ROOT / "data"
RAW_FILE = DATA_DIR / "ECR Analytics Project Data 1.xlsx"
OUT_DIR = ROOT / "outputs" / "00_preliminary"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Analysis parameters
KEEP_STATE = "Closed"
EMERGE_WO_THRESHOLD = 5_000_000
DATE_CUTOFF = pd.Timestamp("2024-01-01")
MY_PLACEHOLDERS = {2035}
MY_START_MONTH, MY_START_YEAR_OFFSET = 7, -1

# Plot colors and style
C_LEGACY, C_EMERGE = "#2a78d6", "#eb6834"
INK, INK_2, GRID = "#0b0b0b", "#52514e", "#e4e3df"
SEQ_BLUES = LinearSegmentedColormap.from_list(
    "seq_blue", ["#f7f9fc", "#cde2fb", "#86b6ef", "#3987e5", "#256abf", "#184f95", "#0d366b"])
ERA_COLORS = {"Legacy": C_LEGACY, "EMERGE": C_EMERGE}

plt.rcParams.update({
    "figure.figsize": (10, 4.5), "figure.dpi": 110, "figure.facecolor": "white",
    "axes.facecolor": "white", "axes.edgecolor": GRID, "axes.labelcolor": INK_2,
    "axes.titlesize": 12, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "axes.grid.axis": "y", "grid.color": GRID, "grid.linewidth": 0.8,
    "xtick.color": INK_2, "ytick.color": INK_2, "text.color": INK,
    "lines.linewidth": 2, "legend.frameon": False,
})

## Step 2: Load the Raw Data
First we load the Excel file and look at its structure before changing anything.

In [ ]:
df_raw = pd.read_excel(RAW_FILE, sheet_name=0)
print(f"{RAW_FILE.name}: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")
df_raw.head(3)

Next we summarize each column: its data type, percent missing, number of unique values, and how often it contains a `|` or `,` delimiter.

In [ ]:
def profile(frame):
    # Build a summary table with one row per column
    rows = []
    for c in frame.columns:
        s = frame[c]
        txt = s.dropna().astype(str)
        rows.append({
            "column": c,
            "dtype": str(s.dtype),
            "null_pct": round(s.isna().mean() * 100, 1),
            "n_unique": s.nunique(dropna=True),
            "pct_with_pipe": round(txt.str.contains(r"\|").mean() * 100, 1) if len(txt) else 0,
            "pct_with_comma": round(txt.str.contains(",").mean() * 100, 1) if len(txt) else 0,
            "example": txt.iloc[0][:60] if len(txt) else None,
        })
    return pd.DataFrame(rows).set_index("column")

raw_profile = profile(df_raw)
raw_profile

We also check for duplicates and the date range of the data.

In [ ]:
print("Duplicate WO_NO :", df_raw["WO_NO"].duplicated().sum())
print("Duplicate EWO_NO:", df_raw["EWO_NO"].duplicated().sum())
print("Fully duplicated rows:", df_raw.duplicated().sum())
print("CREATED_ON range:", df_raw["CREATED_ON"].min(), "→", df_raw["CREATED_ON"].max())

**Observations:**
- Each row is one ECR. There are no duplicate WO_NO or EWO_NO values.
- Many columns hold several values in one cell, separated by `|` (cost areas, cost types, suppliers) or `,` (programs, VPPS, plants). PROGRAMS uses both.
- The `|`-separated values don't line up across columns (for example, "Engineering Impact | Buyer Impact" shows up in both orders), so we treat them as unordered lists.
- Some commas should not be split: they are part of supplier names ("Co.,Ltd") or free text like TITLE and COST_COMMENT.
- SUPPLIER_CONTACT_NAME contains personal contact information (names, emails, phone numbers).

## Step 3: Clean the Data
We work on a copy called `df` so the original `df_raw` stays unchanged.

### 3.1 Clean up text
Remove extra spaces, tabs, and line breaks, and turn blank text into missing values.

In [ ]:
df = df_raw.copy()

text_cols = df.columns[df.dtypes.astype(str).isin(["object", "str", "string"])].tolist()

def normalize_text(s):
    out = (s.astype("string")
             .str.replace(r"[\t\r\n]+", " ", regex=True)
             .str.replace(r"\s{2,}", " ", regex=True)
             .str.strip())
    return out.mask(out.isin(["", "nan", "None", "N/A ", "null"]))

df[text_cols] = df[text_cols].apply(normalize_text)
print(f"Normalized {len(text_cols)} text columns")

### 3.2 Set data types
IDs and counts become integers, CREATED_ON becomes a date, and columns with a small set of repeated values become categories.

In [ ]:
CATEGORICAL_COLS = ["STATE", "TYPE", "SUBTYPE", "REASON", "LEAD_PROGRAM"]

df = df.astype({
    "WO_NO": "Int64",
    "EWO_NO": "string",
    "MODEL_YEAR": "Int64",
    "COUNT_OF_REWORK": "Int64",
})
df["CREATED_ON"] = pd.to_datetime(df["CREATED_ON"], errors="raise")
for c in CATEGORICAL_COLS:
    df[c] = df[c].astype("category")

df.dtypes.to_frame("dtype").T

### 3.3 Convert delimited text to lists
- Columns like COST_TYPE and SUPPLIER_NAME are split on `|` only.
- Code-list columns like PROGRAMS, VPPS, and PLANT are split on both `|` and `,`.
- IS_OVERALL and COST_DELTA_INDICATOR become lists of integers (for example, "0 | 1" becomes [0, 1]).
- TITLE and COST_COMMENT stay as text.

Duplicate values in a list are removed, and missing values become an empty list.

In [ ]:
PIPE = r"\s*\|\s*"
PIPE_OR_COMMA = r"\s*[|,]\s*"

# Column -> which delimiter(s) to split on
LIST_SPEC = {
    "DRIVER": PIPE, "COST_AREA": PIPE, "COST_TYPE": PIPE, "FNA_DESC": PIPE,
    "TOOLING_TYPE": PIPE, "POTENTIAL_NUMBER": PIPE, "SUPPLIER_NAME": PIPE,
    "SUPPLIER_COUNTRY": PIPE,
    "PROGRAMS": PIPE_OR_COMMA, "VPPS": PIPE_OR_COMMA, "VPPS_EXPANDED": PIPE_OR_COMMA,
    "PLANT": PIPE_OR_COMMA, "MANUFACTURING_REGION": PIPE_OR_COMMA,
    "IS_OVERALL": PIPE, "COST_DELTA_INDICATOR": PIPE,
}
INT_LIST_COLS = {"IS_OVERALL", "COST_DELTA_INDICATOR"}


def to_list(value, pattern, as_int=False):
    if pd.isna(value):
        return []
    parts = [p.strip() for p in re.split(pattern, str(value))]
    parts = [p for p in parts if p and p.upper() not in {"N/A", "NA", "NONE"}]
    if as_int:
        parts = [int(p) for p in parts if p.lstrip("-").isdigit()]
    return list(dict.fromkeys(parts))  # remove duplicates, keep order


for col, pattern in LIST_SPEC.items():
    df[col] = df[col].map(lambda v, p=pattern, c=col: to_list(v, p, c in INT_LIST_COLS))

# Compare one row before and after
idx = df_raw["PROGRAMS"].fillna("").str.contains(r"\|").idxmax()
pd.DataFrame({"raw": df_raw.loc[idx, list(LIST_SPEC)], "clean": df.loc[idx, list(LIST_SPEC)]})

Now we add some helper columns, such as the number of programs and suppliers on each ECR.

We also standardize COST_AREA: the old system says "Validation Impact" while EMERGE says "Validation", so we drop the word "Impact".

In [ ]:
# Standardize COST_AREA labels
df["COST_AREA"] = df["COST_AREA"].map(lambda l: list(dict.fromkeys(re.sub(r"\s+Impact$", "", a) for a in l)))

# Count and flag columns
df["N_PROGRAMS"]   = df["PROGRAMS"].str.len()
df["N_SUPPLIERS"]  = df["SUPPLIER_NAME"].str.len()
df["N_COST_TYPES"] = df["COST_TYPE"].str.len()
df["N_FNA"]        = df["FNA_DESC"].str.len()
df["HAS_OVERALL_COST_LINE"] = df["IS_OVERALL"].map(lambda l: 1 in l)
df["HAS_VALIDATION_IMPACT"] = df["COST_AREA"].map(lambda l: "Validation" in l)

# Check: is the LEAD_PROGRAM usually included in PROGRAMS?
lead_in_programs = df.apply(
    lambda r: pd.isna(r["LEAD_PROGRAM"]) or r["LEAD_PROGRAM"] in r["PROGRAMS"] or not r["PROGRAMS"], axis=1)
print(f"LEAD_PROGRAM appears in PROGRAMS list (or one is blank): {lead_in_programs.mean():.1%}")
df[["N_PROGRAMS", "N_SUPPLIERS", "N_COST_TYPES", "N_FNA"]].describe().T

### 3.4 Standardize DRIVER labels
The two systems name drivers differently. For example, the old system uses "EGM" and EMERGE uses "Engineering Group Manager (EGM)". To compare them, we pull out the abbreviation in parentheses and save it in a new column, DRIVER_CODE.

In [ ]:
DRIVER_MANUAL_MAP = {
    "Problem Resolution Tracking System": "PRTS",
}

def canonical_driver(label):
    if label in DRIVER_MANUAL_MAP:
        return [DRIVER_MANUAL_MAP[label]]
    abbrevs = re.findall(r"\(([A-Z][A-Z0-9\-]{1,9})\)", label)
    return abbrevs if abbrevs else [label]

df["DRIVER_CODE"] = df["DRIVER"].map(
    lambda labels: list(dict.fromkeys(code for l in labels for code in canonical_driver(l))))

before = df["DRIVER"].explode().nunique()
after = df["DRIVER_CODE"].explode().nunique()
print(f"Distinct DRIVER values: {before} raw → {after} harmonized")

# Most common label -> code mappings
(df["DRIVER"].explode().dropna().to_frame("raw_label")
   .assign(code=lambda t: t["raw_label"].map(lambda l: ", ".join(canonical_driver(l))))
   .value_counts().head(12).rename("n").reset_index())

### 3.5 Remove personal information
SUPPLIER_CONTACT_NAME contains names, emails, and phone numbers. We don't need it for the analysis, so we replace it with a True/False column showing whether a contact exists.

In [ ]:
df["HAS_SUPPLIER_CONTACT"] = df["SUPPLIER_CONTACT_NAME"].notna()
df = df.drop(columns="SUPPLIER_CONTACT_NAME")

### 3.6 Add new columns
**Date columns:** the year and month each ECR was created.

In [ ]:
df["CREATED_YEAR"] = df["CREATED_ON"].dt.year.astype("Int64")
df["CREATED_YM"]   = df["CREATED_ON"].dt.to_period("M")

**System columns:** two ways to split old vs. new system.
- `SYSTEM_ERA` uses the WO number (Legacy if below 5M, EMERGE otherwise).
- `DATE_ERA` uses a simple date cutoff of January 1, 2024.

In [ ]:
df["SYSTEM_ERA"] = pd.Categorical(
    np.where(df["WO_NO"] >= EMERGE_WO_THRESHOLD, "EMERGE", "Legacy"), categories=["Legacy", "EMERGE"])
df["DATE_ERA"] = pd.Categorical(
    np.where(df["CREATED_ON"] >= DATE_CUTOFF, "2024+", "Pre-2024"), categories=["Pre-2024", "2024+"])

**Rework columns:** COUNT_OF_REWORK is 0 for every EMERGE record, which suggests EMERGE doesn't track rework. We treat it as missing for EMERGE so it doesn't pull the averages down.

In [ ]:
df["REWORK_TRACKED"] = df["SYSTEM_ERA"] == "Legacy"
df["COUNT_OF_REWORK_CLEAN"] = df["COUNT_OF_REWORK"].where(df["REWORK_TRACKED"]).astype("Int64")
df["HAS_REWORK"] = (df["COUNT_OF_REWORK_CLEAN"] > 0).astype("boolean")

**Model year timing columns:** how many months before the model year's production start each ECR was created. A positive number means before launch, and a negative number means after launch.

In [ ]:
df["IS_MY_PLACEHOLDER"] = df["MODEL_YEAR"].isin(MY_PLACEHOLDERS)
my_start = pd.to_datetime(dict(
    year=(df["MODEL_YEAR"] + MY_START_YEAR_OFFSET).astype(float),
    month=MY_START_MONTH, day=1), errors="coerce")
df["MY_START_DATE"] = my_start.where(~df["IS_MY_PLACEHOLDER"])
df["MONTHS_BEFORE_MY_START"] = ((df["MY_START_DATE"] - df["CREATED_ON"]).dt.days / 30.4375).round(1)
df["YEARS_CREATED_BEFORE_MY"] = (df["MODEL_YEAR"] - df["CREATED_YEAR"]).astype("Int64")
df["LAUNCH_PHASE"] = pd.Categorical(
    np.select([df["IS_MY_PLACEHOLDER"], df["MONTHS_BEFORE_MY_START"] >= 0], ["Unknown (MY placeholder)", "Pre-launch"],
              "Post-launch"),
    categories=["Pre-launch", "Post-launch", "Unknown (MY placeholder)"])

df[["WO_NO", "CREATED_ON", "MODEL_YEAR", "SYSTEM_ERA", "DATE_ERA",
    "MY_START_DATE", "MONTHS_BEFORE_MY_START", "LAUNCH_PHASE"]].sample(6, random_state=7)

### 3.7 Check the cleaned data
These checks will stop the notebook with an error if something is wrong, for example if a new data file breaks one of our assumptions.

In [ ]:
assert df["WO_NO"].is_unique, "WO_NO is no longer a unique key"
assert df["EWO_NO"].is_unique, "EWO_NO is no longer a unique key"
assert df["CREATED_ON"].notna().all(), "CREATED_ON has nulls"
assert df["MODEL_YEAR"].notna().all(), "MODEL_YEAR has nulls"
assert (df["COUNT_OF_REWORK"] >= 0).all(), "negative rework count"
for c in LIST_SPEC:
    assert df[c].map(lambda v: isinstance(v, list)).all(), f"{c} not fully converted to lists"
# No WO numbers should fall between the two ranges
gap = df["WO_NO"].between(4_900_000, 6_999_999).sum()
assert gap == 0, f"{gap} WO numbers fall between the legacy and EMERGE ranges — revisit EMERGE_WO_THRESHOLD"
# EMERGE rework should still be all zeros
emerge_rework = df.loc[df["SYSTEM_ERA"] == "EMERGE", "COUNT_OF_REWORK"]
assert (emerge_rework == 0).all(), "EMERGE now reports rework counts — revisit the REWORK_TRACKED assumption"
print("All validation checks passed")
print(f"Clean dataset: {df.shape[0]:,} rows × {df.shape[1]} columns")

## Step 4: Filter to Closed ECRs
First we look at how many ECRs are in each STATE, split by system.

In [ ]:
state_by_era = pd.crosstab(df["STATE"], df["SYSTEM_ERA"], margins=True, margins_name="Total")
state_by_era.sort_values("Total", ascending=False)

Next we keep only the Closed ECRs and count what was removed.

In [ ]:
df_closed = df[df["STATE"] == KEEP_STATE].copy()
df_closed["STATE"] = df_closed["STATE"].cat.remove_unused_categories()

removed = (df.loc[df["STATE"] != KEEP_STATE]
             .groupby(["STATE", "SYSTEM_ERA"], observed=True).size().unstack(fill_value=0))
removed["Total"] = removed.sum(axis=1)
print(f"Kept {len(df_closed):,} of {len(df):,} ECRs ({len(df_closed)/len(df):.1%}); "
      f"removed {len(df) - len(df_closed):,}")
removed.sort_values("Total", ascending=False)

Finally we check what percent of ECRs in each model year are Closed.

In [ ]:
df_closed

In [ ]:
closed_share_my = (df.assign(is_closed=df["STATE"] == KEEP_STATE)
                     .groupby("MODEL_YEAR", observed=True)
                     .agg(n_all=("WO_NO", "size"), n_closed=("is_closed", "sum")))
closed_share_my["closed_pct"] = (closed_share_my["n_closed"] / closed_share_my["n_all"] * 100).round(1)

fig, ax = plt.subplots(figsize=(10, 3.6))
ax.bar(closed_share_my.index.astype(str), closed_share_my["closed_pct"], color=C_LEGACY, width=0.7)
for x, (pct, n) in enumerate(zip(closed_share_my["closed_pct"], closed_share_my["n_all"])):
    ax.text(x, pct + 1.5, f"{pct:.0f}%", ha="center", fontsize=8, color=INK_2)
ax.set_ylim(0, 110); ax.set_ylabel("% of ECRs Closed")
ax.yaxis.set_major_formatter(mticker.PercentFormatter())
ax.set_title("Share of ECRs in Closed state, by MODEL_YEAR")
plt.show()
closed_share_my.T

**Observations:** Legacy ECRs are only ever Closed or Cancelled. The in-progress states (Execute, Review, etc.) only appear in EMERGE. So the Closed filter removes far more of the recent model years (MY2028 and later), and counts for those years should be treated as lower bounds.

## Step 5: Partition by System (Legacy vs. EMERGE)
Here we compare our two ways of splitting the data: the WO number (`SYSTEM_ERA`) and the January 2024 date cutoff (`DATE_ERA`).

In [ ]:
era_summary = (df.groupby("SYSTEM_ERA", observed=True)
                 .agg(n=("WO_NO", "size"),
                      wo_min=("WO_NO", "min"), wo_max=("WO_NO", "max"),
                      first_created=("CREATED_ON", "min"), last_created=("CREATED_ON", "max"),
                      n_states=("STATE", lambda s: s.nunique())))
era_summary

The chart below shows monthly ECR volume by system, to see when the switch to EMERGE happened.

In [ ]:
monthly = (df.groupby(["CREATED_YM", "SYSTEM_ERA"], observed=True).size()
             .unstack(fill_value=0).sort_index())
monthly.index = monthly.index.to_timestamp()
emerge_share = monthly["EMERGE"] / monthly.sum(axis=1)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 6.5), sharex=True, gridspec_kw={"height_ratios": [2, 1.2]})
ax1.bar(monthly.index, monthly["Legacy"], width=25, color=C_LEGACY, label="Legacy (WO < 5M)")
ax1.bar(monthly.index, monthly["EMERGE"], width=25, bottom=monthly["Legacy"], color=C_EMERGE,
        label="EMERGE (WO ≥ 7M)", edgecolor="white", linewidth=0.5)
ax1.axvline(DATE_CUTOFF, color=INK_2, ls="--", lw=1)
ax1.text(DATE_CUTOFF, ax1.get_ylim()[1] * 0.95, "  calendar cutoff 2024-01-01", color=INK_2, fontsize=8, va="top")
ax1.set_ylabel("ECRs created / month"); ax1.legend(loc="upper left")
ax1.set_title("ECR volume by month of creation, by source system")

ax2.plot(emerge_share.index, emerge_share * 100, color=C_EMERGE)
ax2.set_ylabel("% EMERGE"); ax2.set_ylim(-5, 105)
ax2.yaxis.set_major_formatter(mticker.PercentFormatter())
ax2.set_title("EMERGE share of monthly ECRs (migration window)")
plt.tight_layout(); plt.show()

window = emerge_share[(emerge_share > 0) & (emerge_share < 1)]
print(f"Migration window (both systems creating ECRs): {window.index.min():%b %Y} → {window.index.max():%b %Y}")

Next we check how often the date cutoff and the WO-number split agree.

In [ ]:
agree = pd.crosstab(df["DATE_ERA"], df["SYSTEM_ERA"], margins=True)
mismatch = ((df["DATE_ERA"] == "2024+") & (df["SYSTEM_ERA"] == "Legacy")).sum()
print(f"{mismatch:,} ECRs created on/after 2024-01-01 are still legacy records "
      f"({mismatch / (df['DATE_ERA'] == '2024+').sum():.0%} of the '2024+' bucket)")
agree

**Observations:** The switch to EMERGE did not happen on January 1, 2024. Both systems were in use from about August 2024 to May 2025, so the date cutoff labels many legacy ECRs as EMERGE. For the rest of the notebook we use `SYSTEM_ERA`. We still need GM to confirm the official go-live date.

Next we look at how the fields differ between the two systems. The table below shows columns where the percent filled in changes by 5 points or more.

In [ ]:
era = df["SYSTEM_ERA"]

# True/False: is each regular column filled in?
regular = df.drop(columns=list(LIST_SPEC) + ["SYSTEM_ERA"]).notna()
# True/False: is each list column non-empty?
lists = pd.DataFrame({c: df[c].str.len() > 0 for c in LIST_SPEC})

# % filled by system (rows = columns, columns = Legacy / EMERGE)
fill = (pd.concat([regular, lists], axis=1)
          .groupby(era, observed=True).mean()
          .T.astype(float) * 100).round(1)
fill["Δ (pp)"] = (fill["EMERGE"] - fill["Legacy"]).round(1)
fill.loc[fill["Δ (pp)"].abs() >= 5].sort_values("Δ (pp)")

Then we compare the most common DRIVER and REASON values in each system.

In [ ]:
def top_share(frame, col, k=8):
    # Top k values of a column, as a % of each system
    s = frame[[col, "SYSTEM_ERA"]]
    if s[col].map(lambda v: isinstance(v, list)).any():
        s = s.explode(col, ignore_index=True)
    t = pd.crosstab(s[col], s["SYSTEM_ERA"], normalize="columns") * 100
    return t.loc[t.max(axis=1).sort_values(ascending=False).index[:k]].round(1)

print("Raw DRIVER labels — top values by era (% of era)")
display(top_share(df, "DRIVER"))
print("Harmonized DRIVER_CODE — top values by era (% of era)")
display(top_share(df, "DRIVER_CODE"))
print("REASON — top values by era (% of era)")
display(top_share(df, "REASON", 10))

**Observations:** The two systems record several fields differently:
- **DRIVER:** short codes in Legacy, full names in EMERGE. Fixed with DRIVER_CODE (Step 3.4).
- **COST_AREA:** "Impact" was dropped from the labels in EMERGE. Fixed in Step 3.3.
- **COST_TYPE:** EMERGE combines the estimated and negotiated piece and tooling types into one "Piece/Tooling Impact" type. Not fixed, so N_COST_TYPES can't be compared between systems.
- **COUNT_OF_REWORK:** always 0 in EMERGE. Set to missing for EMERGE (Step 3.6).
- **MANUFACTURING_REGION:** filled in much less often in EMERGE.

Last, we compare summary metrics for Closed ECRs in each system.

In [ ]:
era_metrics = (df_closed.groupby("SYSTEM_ERA", observed=True)
                 .agg(n_closed=("WO_NO", "size"),
                      rework_mean=("COUNT_OF_REWORK_CLEAN", "mean"),
                      pct_with_rework=("HAS_REWORK", "mean"),
                      programs_mean=("N_PROGRAMS", "mean"),
                      suppliers_mean=("N_SUPPLIERS", "mean"),
                      cost_types_mean=("N_COST_TYPES", "mean"),
                      pct_validation_impact=("HAS_VALIDATION_IMPACT", "mean")))
for c in ["pct_with_rework", "pct_validation_impact"]:
    era_metrics[c] *= 100
era_metrics.round(2)

## Step 6: Partition by MODEL_YEAR
From here on we use only the Closed ECRs (`df_closed`). First, the number of ECRs in each model year.

In [ ]:
my_era = pd.crosstab(df_closed["MODEL_YEAR"], df_closed["SYSTEM_ERA"])
x = my_era.index.astype(str)

fig, ax = plt.subplots(figsize=(10, 4.2))
ax.bar(x, my_era["Legacy"], color=C_LEGACY, width=0.7, label="Legacy")
ax.bar(x, my_era["EMERGE"], bottom=my_era["Legacy"], color=C_EMERGE, width=0.7,
       label="EMERGE", edgecolor="white", linewidth=1)
for i, tot in enumerate(my_era.sum(axis=1)):
    ax.text(i, tot + 40, f"{tot:,}", ha="center", fontsize=8, color=INK_2)
ax.set_ylabel("Closed ECRs"); ax.legend(loc="upper left")
ax.set_title("Closed ECRs by MODEL_YEAR, by source system")
ax.yaxis.set_major_formatter(mticker.StrMethodFormatter("{x:,.0f}"))
plt.show()

Summary statistics for each model year:

In [ ]:
my_summary = (df_closed.groupby("MODEL_YEAR", observed=True)
                .agg(n=("WO_NO", "size"),
                     pct_emerge=("SYSTEM_ERA", lambda s: (s == "EMERGE").mean() * 100),
                     first_created=("CREATED_ON", "min"),
                     last_created=("CREATED_ON", "max"),
                     median_months_before_start=("MONTHS_BEFORE_MY_START", "median"),
                     pct_post_launch=("LAUNCH_PHASE", lambda s: (s == "Post-launch").mean() * 100),
                     rework_mean_legacy=("COUNT_OF_REWORK_CLEAN", "mean"),
                     pct_with_rework_legacy=("HAS_REWORK", "mean"),
                     programs_mean=("N_PROGRAMS", "mean"),
                     n_lead_programs=("LEAD_PROGRAM", "nunique")))
my_summary["pct_with_rework_legacy"] *= 100
my_summary["first_created"] = my_summary["first_created"].dt.date
my_summary["last_created"] = my_summary["last_created"].dt.date
my_summary.round(1)

The heatmap below shows the mix of ECR reasons in each model year. We only include model years with at least 100 Closed ECRs, and leave out the 2035 placeholder.

In [ ]:
MAIN_MY = my_summary.index[(my_summary["n"] >= 100) & ~my_summary.index.isin(MY_PLACEHOLDERS)]
sub = df_closed[df_closed["MODEL_YEAR"].isin(MAIN_MY)]
top_reasons = sub["REASON"].value_counts().index[:10]
reason_grp = sub["REASON"].astype("string").where(sub["REASON"].isin(top_reasons), "Other")
mix = (pd.crosstab(reason_grp,
                   sub["MODEL_YEAR"], normalize="columns") * 100)
mix = mix.loc[list(top_reasons) + (["Other"] if "Other" in mix.index else [])]

fig, ax = plt.subplots(figsize=(10, 5))
im = ax.imshow(mix.values, cmap=SEQ_BLUES, aspect="auto", vmin=0)
ax.set_xticks(range(mix.shape[1]), mix.columns.astype(str))
ax.set_yticks(range(mix.shape[0]), mix.index)
ax.grid(False)
for (i, j), v in np.ndenumerate(mix.values):
    ax.text(j, i, f"{v:.0f}", ha="center", va="center", fontsize=8,
            color="white" if v > mix.values.max() * 0.55 else INK)
ax.set_title("REASON mix by MODEL_YEAR (% of closed ECRs in each MY)")
fig.colorbar(im, ax=ax, shrink=0.8, label="% of MY")
plt.tight_layout(); plt.show()

Rework count by model year. This uses Legacy ECRs only, since EMERGE doesn't track rework, and only model years with at least 50 Legacy ECRs.

In [ ]:
leg = sub[sub["REWORK_TRACKED"]]
mys = [my for my in MAIN_MY if (leg["MODEL_YEAR"] == my).sum() >= 50]
fig, ax = plt.subplots(figsize=(10, 4))
data = [leg.loc[leg["MODEL_YEAR"] == my, "COUNT_OF_REWORK_CLEAN"].astype(float).values for my in mys]
bp = ax.boxplot(data, tick_labels=[f"{my}\nn={len(d):,}" for my, d in zip(mys, data)], showfliers=False, patch_artist=True,
                medianprops=dict(color=INK, lw=2), widths=0.5)
for b in bp["boxes"]:
    b.set(facecolor="#cde2fb", edgecolor=C_LEGACY)
ax.plot(range(1, len(mys) + 1), [d.mean() for d in data], "o", color=C_EMERGE, ms=7, label="mean")
ax.set_ylabel("COUNT_OF_REWORK"); ax.legend(loc="upper right")
ax.set_title("Rework count per closed legacy ECR, by MODEL_YEAR (outliers hidden)")
plt.show()

## Step 7: CREATED_ON vs. MODEL_YEAR
In this step we look at when ECRs are created compared to their model year. First, a count of ECRs by the year they were created and their model year.

In [ ]:
ct = pd.crosstab(df_closed["CREATED_YEAR"], df_closed["MODEL_YEAR"])

fig, ax = plt.subplots(figsize=(11, 4.2))
im = ax.imshow(ct.values, cmap=SEQ_BLUES, aspect="auto")
ax.set_xticks(range(ct.shape[1]), ct.columns.astype(str))
ax.set_yticks(range(ct.shape[0]), ct.index.astype(str))
ax.set_xlabel("MODEL_YEAR"); ax.set_ylabel("Year ECR created"); ax.grid(False)
for (i, j), v in np.ndenumerate(ct.values):
    if v:
        ax.text(j, i, f"{v:,}", ha="center", va="center", fontsize=7.5,
                color="white" if v > ct.values.max() * 0.55 else INK)
ax.set_title("Closed ECRs: year created × MODEL_YEAR")
fig.colorbar(im, ax=ax, shrink=0.8, label="ECRs")
plt.tight_layout(); plt.show()

Next, how many months before the model year start each ECR was created. Values below the dashed line are ECRs created after launch.

In [ ]:
timed = df_closed[~df_closed["IS_MY_PLACEHOLDER"] & df_closed["MODEL_YEAR"].isin(MAIN_MY)]
fig, ax = plt.subplots(figsize=(10, 4.2))
data = [timed.loc[timed["MODEL_YEAR"] == my, "MONTHS_BEFORE_MY_START"].values for my in MAIN_MY]
bp = ax.boxplot(data, tick_labels=MAIN_MY.astype(str), showfliers=False, patch_artist=True,
                medianprops=dict(color=INK, lw=2), widths=0.5)
for b in bp["boxes"]:
    b.set(facecolor="#cde2fb", edgecolor=C_LEGACY)
ax.axhline(0, color=C_EMERGE, lw=1.5, ls="--")
ax.text(len(MAIN_MY) + 0.4, 0, "MY production start", color=C_EMERGE, fontsize=8, va="bottom", ha="right")
ax.set_ylabel("Months before MY start\n(negative = post-launch)")
ax.set_title("When closed ECRs are created relative to model-year start")
plt.show()

timed.groupby("MODEL_YEAR", observed=True)["MONTHS_BEFORE_MY_START"].describe(percentiles=[.1, .5, .9]).round(1)

The chart below shows the running total of ECRs for each model year as launch gets closer. Lines for future model years stop at the date of the data pull, so they are incomplete.

In [ ]:
extract_date = df["CREATED_ON"].max()
cohort_mys = [my for my in MAIN_MY if 2024 <= my <= 2029]
cmap = plt.get_cmap("Blues")
shades = {my: cmap(0.35 + 0.6 * i / max(len(cohort_mys) - 1, 1)) for i, my in enumerate(cohort_mys)}

fig, ax = plt.subplots(figsize=(11, 5))
bins = np.arange(-36, 61, 1)
for my in cohort_mys:
    m = timed.loc[timed["MODEL_YEAR"] == my, "MONTHS_BEFORE_MY_START"]
    counts = m.value_counts(bins=bins, sort=False).sort_index(ascending=False).cumsum()
    xs = np.array([iv.left for iv in counts.index])
    # stop each line at the data pull date
    censor = (pd.Timestamp(year=int(my) + MY_START_YEAR_OFFSET, month=MY_START_MONTH, day=1) - extract_date).days / 30.4375
    keep = xs >= censor
    ax.plot(xs[keep], counts.values[keep], color=shades[my], label=f"MY{my}")
    ax.annotate(f"MY{my}", (xs[keep][-1], counts.values[keep][-1]), xytext=(4, 0),
                textcoords="offset points", fontsize=8, color=INK_2, va="center")
ax.invert_xaxis()
ax.axvline(0, color=C_EMERGE, ls="--", lw=1.2)
ax.set_xlabel("Months before MY production start  (→ time moves right; right of dashed line = post-launch)")
ax.set_ylabel("Cumulative closed ECRs")
ax.set_title("Cumulative closed ECRs by model-year cohort")
ax.legend(ncol=3, loc="upper left")
ax.grid(axis="x", visible=False)
plt.tight_layout(); plt.show()

Percent of ECRs created before vs. after launch, by model year:

In [ ]:
phase = (pd.crosstab(df_closed["MODEL_YEAR"], df_closed["LAUNCH_PHASE"], normalize="index") * 100).round(1)
phase["n"] = df_closed["MODEL_YEAR"].value_counts().sort_index()
phase

Finally, we check for unusual timing: ECRs created more than 5 years before or more than 2 years after the model year start.

In [ ]:
anom = df_closed[~df_closed["IS_MY_PLACEHOLDER"]]
very_early = anom[anom["MONTHS_BEFORE_MY_START"] > 60]
very_late = anom[anom["MONTHS_BEFORE_MY_START"] < -24]
print(f"Created >5 yrs before MY start: {len(very_early):,}")
print(f"Created >2 yrs after MY start : {len(very_late):,}")
print(f"MY placeholder ({sorted(MY_PLACEHOLDERS)}) closed ECRs: {df_closed['IS_MY_PLACEHOLDER'].sum():,}")
very_late[["WO_NO", "CREATED_ON", "MODEL_YEAR", "LEAD_PROGRAM", "REASON", "TITLE"]].head(8)

## Step 8: Export the Cleaned Data
We save the data as Parquet files, which keep the list columns intact, and also save a CSV version of the Closed data where lists are joined with "; ".

In [ ]:
def for_parquet(frame):
    out = frame.copy()
    out["CREATED_YM"] = out["CREATED_YM"].astype(str)
    return out

def for_flat_file(frame):
    out = for_parquet(frame)
    for c in list(LIST_SPEC) + ["DRIVER_CODE"]:
        out[c] = out[c].map(lambda l: "; ".join(map(str, l)))
    return out

for_parquet(df).to_parquet(OUT_DIR / "ecr_clean_all_states.parquet", index=False)
for_parquet(df_closed).to_parquet(OUT_DIR / "ecr_clean_closed.parquet", index=False)
for_flat_file(df_closed).to_csv(OUT_DIR / "ecr_clean_closed_flat.csv", index=False)

for p in sorted(OUT_DIR.iterdir()):
    print(f"{p.name:35s} {p.stat().st_size / 1e6:6.2f} MB")

## Step 9: Summary of Results

Run every cell above against the extract in `data/`, then record the measured results here: row and column counts, the Closed-filter counts, the Legacy vs. EMERGE split, the MODEL_YEAR and timing findings, and open questions for GM.
